In [1]:
import os
import random
from PIL import Image
import numpy as np
from imgaug import augmenters as iaa

# Compatibility for very new numpy versions
try:
    np.bool
except Exception:
    np.bool = np.bool_

# -------------------------
# Helper: find starting index for augmented filenames
# -------------------------
def find_start_index(folder_path, ext_tuple=(".jpg", ".png", ".jpeg")):
    max_idx = -1

    for fname in os.listdir(folder_path):
        name, ext = os.path.splitext(fname)

        if ext.lower() not in ext_tuple:
            continue

        if "_aug_" in name:
            try:
                idx = int(name.split("_aug_")[-1])
                if idx > max_idx:
                    max_idx = idx
            except:
                pass

    return max_idx + 1

breakhis_augmenter = iaa.Sequential([

    # -----------------------------
    # Geometry (PRIMARY & SAFE)
    # -----------------------------
    iaa.Fliplr(0.5),
    iaa.Flipud(0.5),

    iaa.Affine(
        rotate=(-30, 30),                 # histology-safe
        scale=(0.9, 1.1),                 # small zoom
        shear=(-5, 5),                    # very mild
        translate_percent={"x": (-0.05, 0.05),
                           "y": (-0.05, 0.05)},
        mode="reflect"
    ),

    # -----------------------------
    # Focus robustness (VERY mild)
    # -----------------------------
    iaa.Sometimes(0.2, iaa.OneOf([
        iaa.GaussianBlur(sigma=(0.0, 0.6)),
        iaa.Sharpen(alpha=(0.0, 0.15), lightness=(0.95, 1.05))
    ])),

    # -----------------------------
    # Sensor noise (extremely mild)
    # -----------------------------
    iaa.Sometimes(0.15, iaa.AdditiveGaussianNoise(
        scale=(0, 0.003 * 255)
    )),

], random_order=True)

def get_next_image(available_images, original_images):
    if len(available_images) == 0:
        available_images.extend(original_images)
        random.shuffle(available_images)
    return available_images.pop()

# -------------------------
# Save augmented images
# -------------------------
def save_augmented_images(folder_path, images, augmenter, target_count):
    current_count = len(images)
    to_generate = max(0, target_count - current_count)
    if to_generate == 0:
        return

    # Only ORIGINAL images (never augmented ones)
    original_images = [img for img in images if "_aug_" not in os.path.basename(img)]
    if len(original_images) == 0:
        print(f"No original images found in {folder_path}. Skipping.")
        return

    # Pool for non-repeating selection
    available_images = original_images.copy()
    random.shuffle(available_images)

    image_index = find_start_index(folder_path)

    for _ in range(to_generate):
        try:
            img_path = get_next_image(available_images, original_images)

            img = Image.open(img_path).convert("RGB")
            img_array = np.array(img)

            base_name = os.path.splitext(os.path.basename(img_path))[0]

            augmented = augmenter(image=img_array)

            if augmented is None:
                raise RuntimeError("augmenter returned None")

            if augmented.dtype != np.uint8:
                augmented = np.clip(augmented, 0, 255).astype(np.uint8)

            new_filename = os.path.join(folder_path, f"{base_name}_aug_{image_index}.jpg")

            while os.path.exists(new_filename):
                image_index += 1
                new_filename = os.path.join(folder_path, f"{base_name}_aug_{image_index}.jpg")

            Image.fromarray(augmented).save(new_filename, quality=95)

            image_index += 1

        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            continue

# -------------------------
# Main augment_data function
# -------------------------
def augment_data(dataset_root, target_count_per_class, augmenter=breakhis_augmenter):
    if not os.path.isdir(dataset_root):
        raise ValueError(f"dataset_root not found: {dataset_root}")

    # iterate over classes
    classes = [
        c for c in sorted(os.listdir(dataset_root))
        if os.path.isdir(os.path.join(dataset_root, c))
    ]

    if not classes:
        print("No class folders found.")
        return

    for cls in classes:
        class_path = os.path.join(dataset_root, cls)

        # iterate over types (4 types)
        types = [
            t for t in sorted(os.listdir(class_path))
            if os.path.isdir(os.path.join(class_path, t))
        ]

        if not types:
            print(f"No type folders found in class '{cls}'. Skipping.")
            continue

        for t in types:
            type_path = os.path.join(class_path, t)

            images = [
                os.path.join(type_path, f)
                for f in os.listdir(type_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))
            ]

            current = len(images)

            if current >= target_count_per_class:
                print(f"[SKIP] {cls}/{t}: {current} images")
                continue

            print(f"[AUGMENT] {cls}/{t}: {current} → {target_count_per_class}")
            save_augmented_images(
                folder_path=type_path,
                images=images,
                augmenter=augmenter,
                target_count=target_count_per_class
            )

    print("✅ Data augmentation completed!")

/var/folders/n9/z74ndvz95nn7gy9yw8d4zb0r0000gn/T/ipykernel_4286/558108263.py:9: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  np.bool


In [3]:
augment_data(r"preprocessed_breakhis_dataset/macenko_normalized", target_count_per_class = 1500)

[AUGMENT] Benign/100X: 644 → 1500
[AUGMENT] Benign/200X: 623 → 1500
[AUGMENT] Benign/400X: 588 → 1500
[AUGMENT] Benign/40X: 625 → 1500
[AUGMENT] Malignant/100X: 1437 → 1500
[AUGMENT] Malignant/200X: 1390 → 1500
[AUGMENT] Malignant/400X: 1232 → 1500
[AUGMENT] Malignant/40X: 1370 → 1500
✅ Data augmentation completed!


In [4]:
augment_data(r"preprocessed_breakhis_dataset/reinhard_normalized", target_count_per_class = 1500)

[AUGMENT] Benign/100X: 644 → 1500
[AUGMENT] Benign/200X: 623 → 1500
[AUGMENT] Benign/400X: 588 → 1500
[AUGMENT] Benign/40X: 625 → 1500
[AUGMENT] Malignant/100X: 1437 → 1500
[AUGMENT] Malignant/200X: 1390 → 1500
[AUGMENT] Malignant/400X: 1232 → 1500
[AUGMENT] Malignant/40X: 1370 → 1500
✅ Data augmentation completed!


In [5]:
augment_data(r"preprocessed_breakhis_dataset/ruifrok_normalized", target_count_per_class = 1500)

[AUGMENT] Benign/100X: 644 → 1500
[AUGMENT] Benign/200X: 623 → 1500
[AUGMENT] Benign/400X: 588 → 1500
[AUGMENT] Benign/40X: 625 → 1500
[AUGMENT] Malignant/100X: 1437 → 1500
[AUGMENT] Malignant/200X: 1390 → 1500
[AUGMENT] Malignant/400X: 1232 → 1500
[AUGMENT] Malignant/40X: 1370 → 1500
✅ Data augmentation completed!


In [6]:
augment_data(r"preprocessed_breakhis_dataset/tissue_only_normalized", target_count_per_class = 1500)

[AUGMENT] Benign/100X: 644 → 1500
[AUGMENT] Benign/200X: 623 → 1500
[AUGMENT] Benign/400X: 588 → 1500
[AUGMENT] Benign/40X: 625 → 1500
[AUGMENT] Malignant/100X: 1437 → 1500
[AUGMENT] Malignant/200X: 1390 → 1500
[AUGMENT] Malignant/400X: 1232 → 1500
[AUGMENT] Malignant/40X: 1370 → 1500
✅ Data augmentation completed!


In [7]:
augment_data(r"preprocessed_breakhis_dataset/vahadane_normalized", target_count_per_class = 1500)

[AUGMENT] Benign/100X: 644 → 1500
[AUGMENT] Benign/200X: 623 → 1500
[AUGMENT] Benign/400X: 588 → 1500
[AUGMENT] Benign/40X: 625 → 1500
[AUGMENT] Malignant/100X: 1437 → 1500
[AUGMENT] Malignant/200X: 1390 → 1500
[AUGMENT] Malignant/400X: 1232 → 1500
[AUGMENT] Malignant/40X: 1370 → 1500
✅ Data augmentation completed!
